# PCAOB DeBERTa encoder baseline on Kaggle

1. Create a **private** Kaggle Dataset containing `encoder_kaggle_bundle.zip`.
2. Attach it to this notebook.
3. In Notebook Settings choose **Accelerator → GPU** and enable Internet for the Hugging Face download.
4. Run all cells. The final cell creates `encoder_cloud_results.zip` for download.

In [ ]:
from pathlib import Path
import shutil, zipfile

project = Path('/kaggle/working/selective-audit-routing')
if project.exists():
    shutil.rmtree(project)

input_root = Path('/kaggle/input')
bundles = list(input_root.rglob('encoder_kaggle_bundle.zip'))
configs = list(input_root.rglob('encoder_baseline_v1.json'))
if len(bundles) == 1:
    project.mkdir(parents=True)
    with zipfile.ZipFile(bundles[0]) as archive:
        archive.extractall(project)
elif len(configs) == 1:
    # Kaggle normally expands uploaded ZIP datasets automatically.
    extracted_root = configs[0].parent.parent
    shutil.copytree(extracted_root, project)
else:
    visible = [str(path) for path in input_root.iterdir()]
    raise FileNotFoundError(
        'Attach the private encoder dataset first. '
        f'Found bundles={bundles}, configs={configs}, input entries={visible}'
    )

required = [
    project / 'configs/encoder_baseline_v1.json',
    project / 'src/train_encoder_baseline_v1.py',
    project / 'data/processed/model_dataset_v2.csv',
    project / 'results/tables/model_comparison_test_predictions_v2.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing required files: {missing}'
print('Project ready:', project)

In [ ]:
%pip install -q --upgrade 'transformers>=4.45,<5' sentencepiece scikit-learn pandas
import torch
assert torch.cuda.is_available(), 'GPU is not enabled in Kaggle Settings'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys
command = [sys.executable, str(project / 'src/train_encoder_baseline_v1.py'), '--model-key', 'general_xsmall', '--device', 'cuda', '--cache-dir', '/kaggle/working/hf_cache']
subprocess.run(command, cwd=project, check=True)

In [ ]:
outputs = [
    project / 'results/metrics/encoder_general_xsmall_v1.json',
    project / 'results/reports/encoder_general_xsmall_v1_zh.md',
    project / 'results/tables/encoder_general_xsmall_v1_test_predictions.csv',
]
for path in outputs:
    assert path.exists(), path
archive = shutil.make_archive('/kaggle/working/encoder_cloud_results', 'zip', project / 'results')
print('Download:', archive)